# 🔍 Day 19 — RAG Deep Dive, Vector Databases & Production Q&A System
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Vector Databases & ANN Search | ~11 sec |
| 2 | 10:30–11:00 AM | Advanced RAG Patterns | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Long-Context Strategies & Tool Use | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Production RAG Q&A System Capstone | ~18 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, time, re, json
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — Vector Databases & ANN Search
**Time:** 9:00–10:30 AM | **Run time: ~11 sec**

In [ ]:
# 1.1  Flat (Exact) Vector Index
class FlatIndex:
    """Brute-force exact cosine similarity search — O(n·d) per query."""
    def __init__(self):
        self.vectors = None
        self.metadata = []

    def add(self, vectors, metadata=None):
        # Normalise to unit sphere: cosine sim = dot product after L2-norm
        self.vectors  = normalize(vectors, norm='l2')
        self.metadata = metadata or list(range(len(vectors)))

    def search(self, query, k=5):
        q_norm   = normalize(query.reshape(1,-1), norm='l2')[0]
        scores   = self.vectors @ q_norm          # dot product = cosine sim
        top_k    = np.argsort(scores)[::-1][:k]
        return [(int(idx), float(scores[idx])) for idx in top_k]

    def __len__(self):
        return len(self.vectors) if self.vectors is not None else 0

rng = np.random.default_rng(42)
N, D = 10_000, 128
corpus = rng.normal(0, 1, (N, D))
query  = rng.normal(0, 1, D)

flat_idx = FlatIndex()
flat_idx.add(corpus)

t0 = time.perf_counter()
results = flat_idx.search(query, k=10)
flat_time = (time.perf_counter() - t0) * 1000

print(f'Flat index: {N:,} vectors × {D}D')
print(f'Query time: {flat_time:.2f}ms')
print(f'Top-5 results:')
for idx, score in results[:5]:
    print(f'  idx={idx:5d}  cos_sim={score:.4f}')

In [ ]:
# 1.2  IVF (Inverted File Index) — Cluster-Based Approximate Search
class IVFIndex:
    """
    Inverted File Index:
    1. Train K-means clusters on corpus
    2. Assign each vector to its nearest cluster
    3. At query time: find nearest n_probe clusters, search only those
    """
    def __init__(self, n_clusters=50, n_probe=5):
        self.n_clusters = n_clusters
        self.n_probe    = n_probe
        self.centroids  = None
        self.invlists   = {}   # cluster_id → list of (vector, original_idx)

    def train(self, vectors):
        print(f'Training IVF with {self.n_clusters} clusters...')
        self.vectors_norm = normalize(vectors, norm='l2')
        km = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=3)
        labels = km.fit_predict(self.vectors_norm)
        self.centroids = normalize(km.cluster_centers_, norm='l2')
        # Build inverted lists
        self.invlists = {i: [] for i in range(self.n_clusters)}
        for orig_idx, cluster_id in enumerate(labels):
            self.invlists[cluster_id].append(orig_idx)
        sizes = [len(v) for v in self.invlists.values()]
        print(f'  Avg cluster size: {np.mean(sizes):.0f}  '
              f'Min: {min(sizes)}  Max: {max(sizes)}')

    def search(self, query, k=10):
        q_norm = normalize(query.reshape(1,-1), norm='l2')[0]
        # Step 1: Find nearest n_probe centroids
        centroid_sims = self.centroids @ q_norm
        probe_ids     = np.argsort(centroid_sims)[::-1][:self.n_probe]
        # Step 2: Search only vectors in those clusters
        candidate_idx = []
        for cid in probe_ids:
            candidate_idx.extend(self.invlists[cid])
        if not candidate_idx:
            return []
        cand_vecs  = self.vectors_norm[candidate_idx]
        cand_sims  = cand_vecs @ q_norm
        top_local  = np.argsort(cand_sims)[::-1][:k]
        return [(candidate_idx[i], float(cand_sims[i])) for i in top_local]

ivf = IVFIndex(n_clusters=100, n_probe=10)
ivf.train(corpus)

t0 = time.perf_counter()
results_ivf = ivf.search(query, k=10)
ivf_time = (time.perf_counter() - t0) * 1000
print(f'IVF query time: {ivf_time:.2f}ms  (vs flat: {flat_time:.2f}ms)')

In [ ]:
# 1.3  Recall@K Measurement
def recall_at_k(flat_results, approx_results, k):
    """Fraction of true top-k that appear in approximate top-k."""
    true_ids  = set(idx for idx,_ in flat_results[:k])
    approx_ids = set(idx for idx,_ in approx_results[:k])
    return len(true_ids & approx_ids) / k

# Test over 100 random queries
n_queries = 100
recalls   = []
for _ in range(n_queries):
    q_test  = rng.normal(0, 1, D)
    r_flat  = flat_idx.search(q_test, k=10)
    r_ivf   = ivf.search(q_test, k=10)
    recalls.append(recall_at_k(r_flat, r_ivf, k=10))

print(f'\nIVF Recall@10 over {n_queries} queries:')
print(f'  Mean recall: {np.mean(recalls):.4f}')
print(f'  Min recall : {np.min(recalls):.4f}')
print(f'  Perfect recall: {np.mean(np.array(recalls)==1.0):.1%} of queries')

In [ ]:
# 1.4  Product Quantisation (PQ) — Vector Compression
class ProductQuantiser:
    """
    Split d-dim vector into M sub-vectors of d/M dims each.
    Quantise each sub-vector to one of K centroids.
    Reduces memory from 32*d bits → log2(K)*M bits.
    """
    def __init__(self, d, M=8, K=256):
        self.M = M                    # number of sub-spaces
        self.K = K                    # centroids per sub-space
        self.d_sub = d // M           # dims per sub-space
        self.codebooks = None

    def train(self, X):
        """Learn K centroids for each of M sub-spaces."""
        self.codebooks = []
        for m in range(self.M):
            X_sub = X[:, m*self.d_sub:(m+1)*self.d_sub]
            km    = KMeans(n_clusters=self.K, random_state=42, n_init=1,
                           max_iter=20)
            km.fit(X_sub)
            self.codebooks.append(km.cluster_centers_)

    def encode(self, X):
        """Map each vector to M centroid indices."""
        codes = np.zeros((len(X), self.M), dtype=np.uint8)
        for m in range(self.M):
            X_sub    = X[:, m*self.d_sub:(m+1)*self.d_sub]
            diffs    = X_sub[:, None, :] - self.codebooks[m][None, :, :]
            dists    = np.sum(diffs**2, axis=-1)
            codes[:, m] = dists.argmin(axis=1)
        return codes

    def decode(self, codes):
        """Reconstruct approximate vectors from codes."""
        X_approx = np.zeros((len(codes), self.M * self.d_sub))
        for m in range(self.M):
            X_approx[:, m*self.d_sub:(m+1)*self.d_sub] = \
                self.codebooks[m][codes[:, m].astype(int)]
        return X_approx

# Small demo (reduce d to make PQ fast)
D_pq = 32; M_pq = 4; K_pq = 32
X_pq = rng.normal(0, 1, (500, D_pq))
pq   = ProductQuantiser(d=D_pq, M=M_pq, K=K_pq)
pq.train(X_pq)
codes_pq   = pq.encode(X_pq)
X_pq_recon = pq.decode(codes_pq)
recon_err  = np.mean((X_pq - X_pq_recon)**2)
mem_ratio  = (np.log2(K_pq) * M_pq) / (32 * D_pq)
print(f'Product Quantisation (d={D_pq}, M={M_pq}, K={K_pq}):')
print(f'  Reconstruction MSE : {recon_err:.4f}')
print(f'  Memory ratio       : {mem_ratio:.3f}x  ({mem_ratio:.0%} of original)')

---
## Session 2 — Advanced RAG Patterns
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Basic RAG: Retrieve + Generate
docs_raw = [
    "Random forests build many decision trees using bootstrap sampling and random feature selection.",
    "Gradient boosting trains trees sequentially, each correcting errors from the previous tree.",
    "Neural networks learn hierarchical representations via backpropagation through many layers.",
    "Support vector machines find the maximum-margin hyperplane separating two classes.",
    "K-means clustering minimises within-cluster variance by iterative centroid updates.",
    "Principal component analysis projects data onto orthogonal axes of maximum variance.",
    "Cross-validation estimates generalisation error by rotating train-validation splits.",
    "Regularisation adds a penalty term to the loss to prevent overfitting.",
    "SHAP values measure each feature's marginal contribution across all possible coalitions.",
    "Conformal prediction provides coverage guarantees without distributional assumptions.",
]

tfidf_rag = TfidfVectorizer(ngram_range=(1,2), min_df=1)
D_vecs    = tfidf_rag.fit_transform(docs_raw)

def retrieve(query, k=3, verbose=True):
    q_vec = tfidf_rag.transform([query])
    sims  = cosine_similarity(q_vec, D_vecs)[0]
    top_k = np.argsort(sims)[::-1][:k]
    results = [(docs_raw[i], round(float(sims[i]),4)) for i in top_k]
    if verbose:
        print(f'Query: "{query}"')
        for doc, score in results:
            print(f'  [{score:.4f}] {doc[:60]}...' if len(doc)>60 else f'  [{score:.4f}] {doc}')
    return results

retrieve("how do ensemble models combine weak learners?")

In [ ]:
# 2.2  Cross-Encoder Re-ranking
class CrossEncoderReranker:
    """
    Simulates a cross-encoder: scores query-document pairs jointly.
    In production: BERT or similar processes (query, doc) concatenated.
    Here we simulate with feature overlap + length normalisation.
    """
    def score(self, query, document):
        q_words = set(query.lower().split())
        d_words = set(document.lower().split())
        # Jaccard-like overlap + length bonus
        overlap  = len(q_words & d_words)
        coverage = overlap / (len(q_words) + 1e-9)  # fraction of query covered
        density  = overlap / (len(d_words) + 1e-9)  # precision in doc
        return (2 * coverage * density) / (coverage + density + 1e-9)  # F1-like

reranker = CrossEncoderReranker()

def retrieve_and_rerank(query, first_stage_k=6, final_k=3):
    # Stage 1: Fast retrieval
    candidates = retrieve(query, k=first_stage_k, verbose=False)
    # Stage 2: Re-rank with cross-encoder
    reranked = [(doc, score, reranker.score(query, doc))
                for doc, score in candidates]
    reranked.sort(key=lambda x: x[2], reverse=True)
    return reranked[:final_k]

query_rr = "how does regularisation prevent overfitting?"
print(f'Query: "{query_rr}"')
print('After retrieve-and-rerank:')
for doc, tfidf_s, ce_s in retrieve_and_rerank(query_rr):
    print(f'  [tfidf={tfidf_s:.3f}  ce={ce_s:.4f}] {doc[:65]}')

In [ ]:
# 2.3  HyDE: Hypothetical Document Embedding
def hyde_retrieve(query, hypothetical_answer, k=3):
    """
    HyDE: instead of embedding the query, embed a
    hypothetical answer and use that as the search vector.
    The hypothesis is usually generated by an LLM.
    """
    # Embed the hypothetical answer (richer signal than short query)
    hyp_vec = tfidf_rag.transform([hypothetical_answer])
    sims    = cosine_similarity(hyp_vec, D_vecs)[0]
    top_k   = np.argsort(sims)[::-1][:k]
    return [(docs_raw[i], round(float(sims[i]),4)) for i in top_k]

query_h      = "what reduces model generalisation error?"
hypothetical = ("Regularisation techniques like L1 and L2 add penalty terms "
                "to the loss function to prevent overfitting and improve "
                "generalisation. Cross-validation also helps estimate error.")

print('Standard retrieval:')
for doc, s in retrieve(query_h, k=3, verbose=False):
    print(f'  [{s:.4f}] {doc[:65]}')

print('\nHyDE retrieval (using hypothetical answer):')
for doc, s in hyde_retrieve(query_h, hypothetical, k=3):
    print(f'  [{s:.4f}] {doc[:65]}')

In [ ]:
# 2.4  Query Expansion + Multi-Hop RAG
def query_expand(query, synonyms_map):
    """Expand query with domain synonyms."""
    words = query.lower().split()
    expanded = query
    for word in words:
        if word in synonyms_map:
            expanded += ' ' + ' '.join(synonyms_map[word])
    return expanded

SYNONYMS = {
    'ensemble'   : ['forest', 'boosting', 'bagging', 'combining', 'models'],
    'overfitting': ['regularisation', 'variance', 'generalisation', 'dropout'],
    'features'   : ['variables', 'inputs', 'dimensions', 'attributes'],
    'learn'      : ['train', 'fit', 'optimise', 'gradient'],
    'cluster'    : ['group', 'segment', 'partition', 'kmeans'],
}

query_orig = "how do ensemble methods learn features?"
query_exp  = query_expand(query_orig, SYNONYMS)

results_orig = retrieve(query_orig, k=3, verbose=False)
results_exp  = retrieve(query_exp,  k=3, verbose=False)

print(f'Original query : "{query_orig}"')
print(f'Expanded query : "{query_exp[:80]}..."')
print('\nOriginal results:')
for doc, s in results_orig: print(f'  [{s:.4f}] {doc[:60]}')
print('Expanded results:')
for doc, s in results_exp:  print(f'  [{s:.4f}] {doc[:60]}')

---
## Session 3 — Long-Context Strategies & Tool Use
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Chunking Strategies
long_document = """
Machine learning is a subset of artificial intelligence. It enables computers to learn from
data without explicit programming. Supervised learning requires labelled training examples.
Unsupervised learning discovers hidden patterns without labels. Reinforcement learning trains
agents through reward signals in interactive environments. Deep learning uses neural networks
with many layers to learn hierarchical representations. Convolutional networks excel at image
tasks. Recurrent networks handle sequential data like text and time series. Transformers use
self-attention to capture long-range dependencies efficiently. Fine-tuning adapts pretrained
models to specific downstream tasks with small amounts of labelled data. Prompt engineering
designs effective inputs to elicit desired outputs from language models. Retrieval-augmented
generation combines search with generation for grounded and up-to-date responses.
""".strip()

# Strategy 1: Fixed-size with overlap
def fixed_chunks(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i+chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Strategy 2: Sentence-level
def sentence_chunks(text):
    return [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 15]

# Strategy 3: Recursive (paragraph → sentence)
def recursive_chunks(text, max_words=30):
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    chunks = []
    for para in paragraphs:
        words = para.split()
        if len(words) <= max_words:
            chunks.append(para)
        else:
            for i in range(0, len(words), max_words):
                chunks.append(' '.join(words[i:i+max_words]))
    return chunks

chunks_fixed = fixed_chunks(long_document, chunk_size=30, overlap=5)
chunks_sent  = sentence_chunks(long_document)
chunks_rec   = recursive_chunks(long_document, max_words=25)

print(f'Fixed-size chunks  : {len(chunks_fixed)}  avg_len={np.mean([len(c.split()) for c in chunks_fixed]):.0f} words')
print(f'Sentence chunks    : {len(chunks_sent)}  avg_len={np.mean([len(c.split()) for c in chunks_sent]):.0f} words')
print(f'Recursive chunks   : {len(chunks_rec)}   avg_len={np.mean([len(c.split()) for c in chunks_rec]):.0f} words')

In [ ]:
# 3.2  Map-Reduce Summarisation
def simulate_llm_summarise(text, max_chars=60):
    """Simulate LLM: take first meaningful sentence."""
    sentences = text.split('.')
    for s in sentences:
        s = s.strip()
        if len(s) > 20:
            return s[:max_chars] + ('...' if len(s) > max_chars else '.')
    return text[:max_chars]

def map_reduce_summarise(document, chunk_size=50):
    """Map: summarise each chunk. Reduce: summarise all summaries."""
    words  = document.split()
    chunks = [' '.join(words[i:i+chunk_size])
              for i in range(0, len(words), chunk_size)]
    # MAP: summarise each chunk
    chunk_summaries = [simulate_llm_summarise(c) for c in chunks]
    print(f'Map step: {len(chunks)} chunks → {len(chunk_summaries)} summaries')
    for i, s in enumerate(chunk_summaries):
        print(f'  Chunk {i+1}: {s}')
    # REDUCE: combine summaries
    combined_context = ' '.join(chunk_summaries)
    final_summary    = simulate_llm_summarise(combined_context, max_chars=120)
    print(f'\nFinal summary: {final_summary}')
    return final_summary

final = map_reduce_summarise(long_document)

In [ ]:
# 3.3  Tool Use: Function Calling Pattern
class ToolDispatcher:
    """
    Simulates an LLM function-calling / tool-use layer.
    LLM outputs JSON tool calls; executor runs them.
    """
    def __init__(self):
        self.tools    = {}
        self.call_log = []

    def register(self, name, fn, description, params):
        self.tools[name] = {'fn': fn, 'description': description, 'params': params}

    def dispatch(self, tool_call):
        """tool_call = {'name': ..., 'args': {...}}"""
        name = tool_call.get('name')
        args = tool_call.get('args', {})
        if name not in self.tools:
            return {'error': f'Unknown tool: {name}', 'tool': name}
        try:
            result = self.tools[name]['fn'](**args)
            entry  = {'tool': name, 'args': args, 'result': result, 'status': 'ok'}
        except Exception as e:
            entry  = {'tool': name, 'args': args, 'error': str(e), 'status': 'error'}
        self.call_log.append(entry)
        return entry

    def schema(self):
        return {name: {'description': t['description'], 'params': t['params']}
                for name, t in self.tools.items()}

# Register tools
td = ToolDispatcher()
td.register('search_kb', lambda query: retrieve(query, k=2, verbose=False),
            'Search the ML knowledge base', {'query': 'str'})
td.register('calculate', lambda expr: eval(expr),
            'Evaluate a mathematical expression', {'expr': 'str'})
td.register('count_words', lambda text: len(text.split()),
            'Count words in a string', {'text': 'str'})
td.register('retrieve_chunk', lambda doc_id: docs_raw[doc_id] if doc_id < len(docs_raw) else 'Not found',
            'Retrieve document by ID', {'doc_id': 'int'})

print('Registered tools:')
for name, info in td.schema().items():
    print(f'  {name}: {info["description"]}')
print()

In [ ]:
# 3.4  ReAct Agent Loop (Reason → Act → Observe)
class ReActAgent:
    """
    ReAct pattern: the agent reasons, acts (calls a tool),
    observes the result, and repeats until it has an answer.
    """
    def __init__(self, dispatcher, max_steps=5):
        self.dispatcher = dispatcher
        self.max_steps  = max_steps

    def run(self, task):
        print(f'Task: "{task}"\n')
        thoughts  = []
        for step in range(self.max_steps):
            # Simulate reasoning step
            thought = self._reason(task, thoughts)
            print(f'Step {step+1} — Thought: {thought["thought"]}')

            if thought['action'] == 'ANSWER':
                print(f'Final answer: {thought["content"]}')
                return thought['content']

            # Act: call the tool
            result = self.dispatcher.dispatch(
                {'name': thought['tool'], 'args': thought['args']}
            )
            print(f'         Act : {thought["tool"]}({thought["args"]})')
            print(f'         Obs : {str(result["result"])[:80]}')
            thoughts.append({'thought': thought, 'observation': result})
            print()

        return 'Max steps reached'

    def _reason(self, task, history):
        """Simulate LLM reasoning step."""
        step = len(history)
        if step == 0:
            return {'thought': 'I should search for relevant documents first.',
                    'action': 'TOOL', 'tool': 'search_kb',
                    'args': {'query': task}}
        elif step == 1:
            obs = history[0]['observation']['result']
            if obs:
                doc, score = obs[0]
                return {'thought': f'Found relevant doc (score={score}). Count its words.',
                        'action': 'TOOL', 'tool': 'count_words',
                        'args': {'text': doc}}
        elif step >= 2:
            obs1 = history[0]['observation']['result']
            doc_text = obs1[0][0] if obs1 else 'No document found'
            return {'thought': 'I have enough information to answer.',
                    'action': 'ANSWER',
                    'content': f'Based on: "{doc_text[:80]}..."'}
        return {'thought': 'Answering directly.', 'action': 'ANSWER',
                'content': 'No specific answer found.'}

agent = ReActAgent(td)
agent.run("How do ensemble methods work?")

---
## Lunch Break — 1:00–2:00 PM
1. What is the difference between IVF and HNSW indexing strategies?
2. When does HyDE improve over standard query-embedding retrieval?
3. What is the key difference between map-reduce and recursive summarisation?
4. Why is the ReAct pattern more reliable than single-shot tool calling?
---

## Session 5 — Production RAG Q&A System Capstone
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Build Production Knowledge Base
KNOWLEDGE_BASE = [
    {"id": 0, "title": "Random Forests",
     "text": "Random forests build many decision trees using bootstrap sampling. Each tree is trained on a random subset of features. Final predictions are made by majority vote or averaging across all trees."},
    {"id": 1, "title": "Gradient Boosting",
     "text": "Gradient boosting builds trees sequentially. Each new tree corrects the residual errors of all previous trees. XGBoost and LightGBM are fast implementations using histogram-based splitting."},
    {"id": 2, "title": "Neural Networks",
     "text": "Neural networks use backpropagation to compute gradients layer by layer. Activation functions introduce non-linearity. Batch normalisation stabilises training. Dropout prevents overfitting."},
    {"id": 3, "title": "SVM",
     "text": "Support vector machines find the maximum-margin hyperplane between classes. The kernel trick maps data to higher dimensions implicitly. SVMs work well in high-dimensional spaces with few samples."},
    {"id": 4, "title": "K-Means",
     "text": "K-means assigns each point to its nearest centroid. Centroids are recomputed as cluster means. The algorithm minimises within-cluster sum of squared distances."},
    {"id": 5, "title": "PCA",
     "text": "PCA finds orthogonal directions of maximum variance. The first principal component captures most variance. SVD is used to compute principal components efficiently."},
    {"id": 6, "title": "Regularisation",
     "text": "L1 regularisation (Lasso) produces sparse weights. L2 regularisation (Ridge) shrinks weights towards zero. Elastic Net combines both. Dropout is a regularisation technique for neural networks."},
    {"id": 7, "title": "Cross-Validation",
     "text": "K-fold cross-validation splits data into K folds. Each fold serves as validation once. Stratified CV maintains class balance. Nested CV prevents data leakage during hyperparameter search."},
    {"id": 8, "title": "SHAP Values",
     "text": "SHAP values measure each feature's contribution across all possible coalitions. TreeSHAP is exact for tree models. SHAP values satisfy additivity and symmetry properties."},
    {"id": 9, "title": "Conformal Prediction",
     "text": "Conformal prediction provides coverage guarantees without distributional assumptions. Nonconformity scores are calibrated on a held-out set. Prediction sets include all plausible labels."},
    {"id": 10, "title": "Federated Learning",
     "text": "Federated learning trains models across distributed clients without sharing raw data. FedAvg aggregates local model updates. Differential privacy adds noise to protect individual contributions."},
    {"id": 11, "title": "Attention Mechanism",
     "text": "Attention computes query-key dot products scaled by sqrt(d_k). Softmax produces attention weights. Multi-head attention runs parallel attention heads then concatenates outputs."},
]

print(f'Knowledge base: {len(KNOWLEDGE_BASE)} documents')
for doc in KNOWLEDGE_BASE[:3]:
    print(f'  [{doc["id"]}] {doc["title"]}: {doc["text"][:60]}...')

In [ ]:
# 5.2  Index the Knowledge Base
# Build TF-IDF + dense SVD embeddings
texts       = [d['text'] for d in KNOWLEDGE_BASE]
titles      = [d['title'] for d in KNOWLEDGE_BASE]

# TF-IDF sparse vectors
tfidf_kb = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X_tfidf  = tfidf_kb.fit_transform(texts)

# Dense embeddings via TruncatedSVD on TF-IDF
svd_kb  = TruncatedSVD(n_components=16, random_state=42)
X_dense = svd_kb.fit_transform(X_tfidf)
X_dense = normalize(X_dense, norm='l2')  # unit sphere

print(f'TF-IDF index  : {X_tfidf.shape}')
print(f'Dense index   : {X_dense.shape}  (SVD-16 on TF-IDF)')

In [ ]:
# 5.3  Hybrid Retriever: Sparse + Dense
class HybridRetriever:
    """
    Combines sparse (TF-IDF) and dense (SVD) retrieval.
    Final score = alpha * sparse_score + (1-alpha) * dense_score
    """
    def __init__(self, tfidf_vec, svd_model, X_tfidf, X_dense,
                 documents, alpha=0.5):
        self.tfidf   = tfidf_vec
        self.svd     = svd_model
        self.X_tfidf = X_tfidf
        self.X_dense = X_dense
        self.docs    = documents
        self.alpha   = alpha

    def retrieve(self, query, k=5, reranker=None):
        # Sparse score
        q_sparse = self.tfidf.transform([query])
        sparse_s = cosine_similarity(q_sparse, self.X_tfidf)[0]

        # Dense score
        q_dense  = normalize(self.svd.transform(q_sparse), norm='l2')
        dense_s  = self.X_dense @ q_dense[0]

        # Hybrid
        hybrid_s = self.alpha * sparse_s + (1-self.alpha) * dense_s
        top_k    = np.argsort(hybrid_s)[::-1][:k]

        results  = [(self.docs[i], float(hybrid_s[i]),
                     float(sparse_s[i]), float(dense_s[i]))
                    for i in top_k]

        # Optional re-ranking
        if reranker:
            results = sorted(results,
                             key=lambda r: reranker.score(query, r[0]['text']),
                             reverse=True)
        return results

hybrid = HybridRetriever(tfidf_kb, svd_kb, X_tfidf, X_dense, KNOWLEDGE_BASE)

test_queries = [
    "how do trees correct each other in boosting?",
    "what prevents neural networks from memorising training data?",
    "how does attention work in transformers?",
]

for q in test_queries:
    results = hybrid.retrieve(q, k=2)
    print(f'Q: "{q}"')
    for doc, hyb, sp, de in results:
        print(f'  [{hyb:.4f}] {doc["title"]}')
    print()

In [ ]:
# 5.4  RAG Evaluation: MRR, Recall@K, Faithfulness
def mean_reciprocal_rank(queries_with_relevant, retriever, k=5):
    """MRR: mean of 1/rank of first relevant result."""
    rrs = []
    for query, relevant_ids in queries_with_relevant:
        results = retriever.retrieve(query, k=k)
        rr = 0.0
        for rank, (doc, *_) in enumerate(results, 1):
            if doc['id'] in relevant_ids:
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return float(np.mean(rrs))

def recall_at_k_rag(queries_with_relevant, retriever, k=5):
    """Recall@K: fraction of relevant docs in top-k."""
    recalls = []
    for query, relevant_ids in queries_with_relevant:
        results    = retriever.retrieve(query, k=k)
        retrieved  = {doc['id'] for doc, *_ in results}
        recall     = len(retrieved & set(relevant_ids)) / len(relevant_ids)
        recalls.append(recall)
    return float(np.mean(recalls))

def faithfulness_score(answer, context):
    """Simple faithfulness: fraction of answer n-grams in context."""
    ans_words = set(answer.lower().split())
    ctx_words = set(context.lower().split())
    overlap   = ans_words & ctx_words
    return len(overlap) / (len(ans_words) + 1e-9)

# Test queries with known relevant documents
eval_queries = [
    ("how do ensemble tree methods reduce variance?", [0, 1]),
    ("what stops neural networks overfitting?",       [2, 6]),
    ("explain attention mechanism in transformers",   [11]),
    ("how does k-means minimise cluster distances?",  [4]),
    ("what guarantees does conformal prediction give?",[9]),
]

mrr = mean_reciprocal_rank(eval_queries, hybrid, k=5)
rcl = recall_at_k_rag(eval_queries, hybrid, k=5)

print(f'RAG Retrieval Evaluation:')
print(f'  MRR @5    : {mrr:.4f}')
print(f'  Recall@5  : {rcl:.4f}')

# Per-query
print(f'\nPer-query breakdown:')
for query, relevant in eval_queries:
    results = hybrid.retrieve(query, k=5)
    top_ids = [doc['id'] for doc,*_ in results]
    hit     = any(r in top_ids for r in relevant)
    rank    = next((i+1 for i,(doc,*_) in enumerate(results) if doc['id'] in relevant), None)
    print(f'  ["{query[:40]}"]: hit={hit}  rank={rank}  top=[{top_ids[:3]}]')

In [ ]:
# 5.5  Full Production RAG Pipeline + Answer Generation
def rag_pipeline(query, retriever, k=3, max_context_chars=400):
    """
    Full RAG pipeline:
    1. Retrieve top-k documents
    2. Build context string (truncated to max_context_chars)
    3. Build prompt
    4. Simulate answer generation
    5. Compute faithfulness score
    """
    # Step 1: Retrieve
    results = retriever.retrieve(query, k=k)

    # Step 2: Build context
    context_parts = []
    for doc, score, *_ in results:
        context_parts.append(f"[{doc['title']}] {doc['text']}")
    context = ' | '.join(context_parts)[:max_context_chars]

    # Step 3: Prompt
    prompt = f"""Use the context below to answer the question.

Context: {context}

Question: {query}

Answer:"""

    # Step 4: Simulate answer (extract first relevant sentence from context)
    answer = simulate_llm_summarise(context, max_chars=120)

    # Step 5: Faithfulness
    faith = faithfulness_score(answer, context)

    return {
        'query'          : query,
        'retrieved'      : [(d['title'], round(s,4)) for d,s,*_ in results],
        'context_chars'  : len(context),
        'answer'         : answer,
        'faithfulness'   : round(faith, 4),
    }

print('=== Production RAG Q&A System ===')
test_qs = [
    "how do random forests make predictions?",
    "what is backpropagation in neural networks?",
    "explain principal component analysis",
]

for q in test_qs:
    result = rag_pipeline(q, hybrid, k=3)
    print(f'\nQ: {result["query"]}')
    print(f'Retrieved: {result["retrieved"]}')
    print(f'Answer: {result["answer"]}')
    print(f'Faithfulness: {result["faithfulness"]:.4f}')

In [ ]:
# 5.6  Visualisation: Retrieval Quality + Embedding Space
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Embedding space (PCA 2D of dense embeddings)
from sklearn.decomposition import PCA
pca2d = PCA(n_components=2, random_state=42)
X_2d  = pca2d.fit_transform(X_dense)

for i, (doc, (x, y)) in enumerate(zip(KNOWLEDGE_BASE, X_2d)):
    axes[0].scatter(x, y, s=80, color='#534AB7', alpha=0.8)
    axes[0].annotate(doc['title'], (x,y), xytext=(4,4),
                     textcoords='offset points', fontsize=7)
axes[0].set_title('Knowledge Base Embedding Space (PCA 2D)')
axes[0].grid(alpha=0.3)

# MRR & Recall across different k values
k_vals  = [1, 2, 3, 5]
mrrs    = [mean_reciprocal_rank(eval_queries, hybrid, k=k) for k in k_vals]
recalls = [recall_at_k_rag(eval_queries, hybrid, k=k)      for k in k_vals]

axes[1].plot(k_vals, mrrs,    'o-', color='#534AB7', linewidth=2, markersize=7, label='MRR@k')
axes[1].plot(k_vals, recalls, 's-', color='#1D9E75', linewidth=2, markersize=7, label='Recall@k')
axes[1].set_xlabel('k (number of retrieved docs)')
axes[1].set_ylabel('Score')
axes[1].set_title('RAG Retrieval Quality vs k')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Production RAG System: Embedding Space + Retrieval Quality', fontsize=12)
plt.tight_layout(); plt.show()

---
## Day 19 Summary

| Concept | Skill Gained |
|---|---|
| Flat exact index | Brute-force cosine similarity, O(n·d) search |
| IVF index | K-means clusters, probe n_probe clusters at query time |
| Recall@K | Fraction of true top-k in approximate top-k results |
| Product quantisation | M sub-spaces × K centroids, 4x memory compression |
| Cross-encoder re-ranking | Second-stage joint query-document scoring |
| HyDE | Embed hypothetical answer instead of raw query |
| Query expansion | Synonym injection to increase retrieval recall |
| Fixed/sentence/recursive chunking | Three splitting strategies compared |
| Map-reduce summarisation | Chunk → summarise → combine summaries |
| Tool dispatcher | JSON function calling, register + dispatch pattern |
| ReAct agent | Reason → Act → Observe loop with tool integration |
| Hybrid retriever | α × sparse + (1-α) × dense score fusion |
| MRR & Recall@K | RAG retrieval evaluation metrics |
| Faithfulness score | Answer n-gram overlap with retrieved context |

---
## Day 20 Preview
- Model serving at scale: gRPC, async batching, model parallelism
- Advanced monitoring: data drift, model drift, alert pipelines
- Cost optimisation: quantisation, distillation, caching strategies
- Final capstone: design and build a production-grade ML system from scratch

In [ ]:
# Bonus: Day 19 in one cell
rng_b = np.random.default_rng(0)
N_b, D_b = 1000, 32
corpus_b = rng_b.normal(0,1,(N_b,D_b))
q_b      = rng_b.normal(0,1,D_b)
idx_b    = FlatIndex(); idx_b.add(corpus_b)
top_b, sc_b = zip(*idx_b.search(q_b, k=3))
print(f'Flat search top-3: {list(top_b)}  scores: {[round(s,4) for s in sc_b]}')

results_b = hybrid.retrieve("how does attention work?", k=2)
for doc, hyb, *_ in results_b:
    print(f'Hybrid RAG: [{hyb:.4f}] {doc["title"]}')

print(f'\nRAG MRR@5  : {mrr:.4f}')
print(f'RAG Recall@5: {rcl:.4f}')
print('\nDay 19 complete — vector databases, ANN, re-ranking, tool use, RAG capstone!')